<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/13_the_layer_clustered_data_router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Environment & Paths
MODEL_PATH = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
INPUT_JSON_PATH = "/content/drive/MyDrive/ResearchProject/trace_map.json"
MIN_CLUSTER_SIZE = 16  # Minimum batching threshold to prevent layer overfitting
BATCH_SIZE = 4         # Safe batch size for Colab A100

# 2. PyTorch Dataset Class
class MUSE_Dataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze()
        }

# 3. Load Tokenizer
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Load and Route the JSON Data
print("\n--- Initializing Data Router ---")
with open(INPUT_JSON_PATH, "r") as f:
    trace_map = json.load(f)

# Initialize raw cluster bins (32 layers)
raw_clusters = {layer: [] for layer in range(32)}

for article_id, data in trace_map.items():
    # Strategy 3 groups by the PRIMARY (index 0) layer responsible for the fact
    primary_layer = data['top_layers'][0]
    raw_clusters[primary_layer].append(data['text'])

# 5. Filter and Build DataLoaders
cluster_dataloaders = {}
residual_texts = []

print("\n📊 Cluster Distribution Analysis:")
print("-" * 40)
print(f"{'Layer ID':<10} | {'Total Articles':<15} | {'Status'}")
print("-" * 40)

for layer, texts in raw_clusters.items():
    count = len(texts)

    if count == 0:
        continue # Skip empty layers entirely

    if count >= MIN_CLUSTER_SIZE:
        print(f"Layer {layer:<5} | {count:<15} | ✅ Validated for Batched QAT")
        dataset = MUSE_Dataset(texts, tokenizer)
        # We shuffle to ensure batches within the cluster are randomized
        cluster_dataloaders[layer] = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    else:
        print(f"Layer {layer:<5} | {count:<15} | ⚠️ Too small. Moved to Residuals.")
        residual_texts.extend(texts)

# 6. Handle the Residual Dataset
if len(residual_texts) > 0:
    print("-" * 40)
    print(f"Residuals  | {len(residual_texts):<15} | 🔄 Grouped into catch-all loader")
    residual_dataset = MUSE_Dataset(residual_texts, tokenizer)
    residual_dataloader = DataLoader(residual_dataset, batch_size=BATCH_SIZE, shuffle=True)
else:
    residual_dataloader = None

print("\n✅ Data Routing Complete!")
print(f"Total Valid Layer Clusters: {len(cluster_dataloaders)}")

Loading Tokenizer...

--- Initializing Data Router ---

📊 Cluster Distribution Analysis:
----------------------------------------
Layer ID   | Total Articles  | Status
----------------------------------------
Layer 0     | 92              | ✅ Validated for Batched QAT
Layer 1     | 176             | ✅ Validated for Batched QAT
Layer 2     | 143             | ✅ Validated for Batched QAT
Layer 3     | 16              | ✅ Validated for Batched QAT
Layer 4     | 20              | ✅ Validated for Batched QAT
Layer 5     | 18              | ✅ Validated for Batched QAT
Layer 6     | 59              | ✅ Validated for Batched QAT
Layer 7     | 37              | ✅ Validated for Batched QAT
Layer 8     | 18              | ✅ Validated for Batched QAT
Layer 9     | 10              | ⚠️ Too small. Moved to Residuals.
Layer 10    | 14              | ⚠️ Too small. Moved to Residuals.
Layer 11    | 9               | ⚠️ Too small. Moved to Residuals.
Layer 12    | 5               | ⚠️ Too small. Moved t